<a href="https://colab.research.google.com/github/Karthikrithvi18/project1/blob/main/Data_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)


In [2]:
# Import the requests library to handle HTTP GET requests
import requests

# Define the base URL endpoint for the Open-Meteo forecast API
url = "https://api.open-meteo.com/v1/forecast"

# Create a dictionary to hold the specific query parameters for our request
params = {
    # Set the latitude coordinate for Berlin
    "latitude": 52.52,
    # Set the longitude coordinate for Berlin
    "longitude": 13.41,
    # Specify a comma-separated list of the hourly weather variables we want
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    # Limit the returned data to exactly the next 24 hours
    "forecast_hours": 24
# Close the parameters dictionary
}

# Send the GET request to the API with the URL and parameters, and store the result
response = requests.get(url, params=params)

# Extract the raw JSON text string directly from the response object
raw_data = response.text

# Print the raw, unformatted text response to the console
print(raw_data)


{"latitude":52.512604,"longitude":13.419517,"generationtime_ms":0.10526180267333984,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":38.0,"hourly_units":{"time":"iso8601","temperature_2m":"°C","relative_humidity_2m":"%","wind_speed_10m":"km/h"},"hourly":{"time":["2026-11-03T07:00","2026-11-03T08:00","2026-11-03T09:00","2026-11-03T10:00","2026-11-03T11:00","2026-11-03T12:00","2026-11-03T13:00","2026-11-03T14:00","2026-11-03T15:00","2026-11-03T16:00","2026-11-03T17:00","2026-11-03T18:00","2026-11-03T19:00","2026-11-03T20:00","2026-11-03T21:00","2026-11-03T22:00","2026-11-03T23:00","2026-11-04T00:00","2026-11-04T01:00","2026-11-04T02:00","2026-11-04T03:00","2026-11-04T04:00","2026-11-04T05:00","2026-11-04T06:00"],"temperature_2m":[4.2,5.1,6.3,7.8,9.1,9.9,10.1,9.9,9.3,8.3,7.8,7.6,7.6,8.0,8.2,8.3,7.8,7.2,6.6,6.0,5.3,5.3,4.9,4.9],"relative_humidity_2m":[81,80,77,70,63,58,58,57,61,67,75,81,87,91,92,90,90,89,90,90,92,89,90,87],"wind_speed_10m":[7.1,7.1,7.1,7.9,

In [3]:
# Import the pandas library for data manipulation
import pandas as pd

# Parse the JSON response into a Python dictionary
data = response.json()

# Create a pandas DataFrame directly from the 'hourly' dictionary
df = pd.DataFrame(data['hourly'])

# Rename the columns to clearer, specified names
df = df.rename(columns={
    "temperature_2m": "temperature_c",
    "relative_humidity_2m": "humidity_pct",
    "wind_speed_10m": "wind_speed_kmh"
})

# Display the first few rows of the formatted DataFrame
print(df.head())


               time  temperature_c  humidity_pct  wind_speed_kmh
0  2026-11-03T07:00            4.2            81             7.1
1  2026-11-03T08:00            5.1            80             7.1
2  2026-11-03T09:00            6.3            77             7.1
3  2026-11-03T10:00            7.8            70             7.9
4  2026-11-03T11:00            9.1            63             8.4


In [4]:
# Check for missing values in the specific columns and count them
missing_counts = df[['temperature_c', 'humidity_pct', 'wind_speed_kmh']].isna().sum()

# Print the count of missing values for each column to the console
print("Missing values per column:\n", missing_counts)

# Drop any rows from the DataFrame where missing values exist in the subset of specified columns
df = df.dropna(subset=['temperature_c', 'humidity_pct', 'wind_speed_kmh'])

# Print a confirmation message along with the new number of rows in the DataFrame
print(f"Data cleaning complete. The DataFrame now has {len(df)} rows.")


Missing values per column:
 temperature_c     0
humidity_pct      0
wind_speed_kmh    0
dtype: int64
Data cleaning complete. The DataFrame now has 24 rows.


In [5]:
# Print the current data types of all columns in the DataFrame
print("Original data types:\n", df.dtypes)

# Convert the temperature, humidity, and wind speed columns to float data types simultaneously
df[['temperature_c', 'humidity_pct', 'wind_speed_kmh']] = df[['temperature_c', 'humidity_pct', 'wind_speed_kmh']].astype(float)

# Convert the 'time' column from a string representation to proper pandas datetime objects
df['time'] = pd.to_datetime(df['time'])

# Print the updated data types to verify the conversions were successful
print("\nUpdated data types:\n", df.dtypes)


Original data types:
 time               object
temperature_c     float64
humidity_pct        int64
wind_speed_kmh    float64
dtype: object

Updated data types:
 time              datetime64[ns]
temperature_c            float64
humidity_pct             float64
wind_speed_kmh           float64
dtype: object


In [6]:
# Count the number of duplicate rows based specifically on the 'time' column
duplicate_count = df.duplicated(subset=['time']).sum()

# Print the number of duplicate rows found to the console
print(f"Number of duplicate rows found: {duplicate_count}")

# Drop any duplicate rows based on the 'time' column, keeping only the first occurrence
df = df.drop_duplicates(subset=['time'], keep='first')

# Print a confirmation message with the updated number of rows in the DataFrame
print(f"Duplicate removal complete. The DataFrame now has {len(df)} rows.")


Number of duplicate rows found: 0
Duplicate removal complete. The DataFrame now has 24 rows.


In [7]:
# Create a boolean mask to identify impossible temperature values (below absolute zero: -273.15 C)
invalid_temp = df['temperature_c'] < -273.15

# Create a boolean mask for impossible relative humidity values (below 0% or above 100%)
invalid_hum = (df['humidity_pct'] < 0) | (df['humidity_pct'] > 100)

# Create a boolean mask for impossible wind speeds (negative values)
invalid_wind = df['wind_speed_kmh'] < 0

# Combine the individual masks to flag any row containing at least one physically impossible value
invalid_mask = invalid_temp | invalid_hum | invalid_wind

# Extract the rows that contain physically impossible values using the combined mask
invalid_rows = df[invalid_mask]

# Print the identified invalid rows to the console for inspection
print("Physically impossible rows found:\n", invalid_rows)

# Update the DataFrame by keeping only the valid rows (inverting the invalid mask)
df = df[~invalid_mask]

# Print a confirmation message with the updated number of rows in the DataFrame
print(f"Filtering complete. The DataFrame now has {len(df)} rows.")


Physically impossible rows found:
 Empty DataFrame
Columns: [time, temperature_c, humidity_pct, wind_speed_kmh]
Index: []
Filtering complete. The DataFrame now has 24 rows.


In [8]:
# Import the gspread library and its specific exception classes for handling missing sheets/tabs
import gspread
from gspread.exceptions import SpreadsheetNotFound, WorksheetNotFound

# (Assuming 'gc' is your already authenticated gspread client object)

# Attempt to open the 'weather_report' spreadsheet if it already exists in your Drive
try:
    # Open the existing spreadsheet by its title
    sh = gc.open('weather_report')
    # Print a confirmation that the existing file was found and opened
    print("Opened existing spreadsheet 'weather_report'.")

# Handle the specific error that occurs if the spreadsheet does not exist
except SpreadsheetNotFound:
    # Create a brand new spreadsheet named 'weather_report'
    sh = gc.create('weather_report')
    # Print a confirmation that a new file was created
    print("Created new spreadsheet 'weather_report'.")

# Attempt to select the worksheet tab named 'Output'
try:
    # Select the existing worksheet tab
    worksheet = sh.worksheet('Output')
    # Print a confirmation that the tab was found
    print("Opened existing worksheet 'Output'.")

# Handle the error if the 'Output' tab does not exist yet
except WorksheetNotFound:
    # Add a new worksheet tab named 'Output' with a default size (100 rows, 20 columns)
    worksheet = sh.add_worksheet(title='Output', rows=100, cols=20)
    # Print a confirmation that the new tab was created
    print("Created new worksheet 'Output'.")

    # Define the exact header row names as a list of strings
    headers = ['time', 'temperature_c', 'wind_speed_kmh', 'humidity_pct']
    # Append the header row to the very top of the newly created worksheet
    worksheet.append_row(headers)
    # Print a confirmation that the headers were written
    print("Added header row to the newly created 'Output' worksheet.")


Created new spreadsheet 'weather_report'.
Created new worksheet 'Output'.
Added header row to the newly created 'Output' worksheet.


In [9]:
print(sh.url)

https://docs.google.com/spreadsheets/d/17HwTPG4pbFrLyjR_1NSqkPXyZIDB-0KhgFNEuAbKeUA


In [10]:
# Fetch the existing header names from the very first row of the worksheet
sheet_headers = worksheet.row_values(1)

# Reorder the DataFrame columns to exactly match the sequence of the worksheet headers
df_aligned = df[sheet_headers].copy()

# Convert the datetime 'time' column to a string format so it can be serialized and sent to the Google API
df_aligned['time'] = df_aligned['time'].astype(str)

# Convert the aligned pandas DataFrame into a standard Python list of lists expected by gspread
data_to_append = df_aligned.values.tolist()

# Append all the rows at once to the next available empty row in the worksheet
worksheet.append_rows(data_to_append)

# Print a confirmation message showing how many rows were successfully appended to the sheet
print(f"Successfully appended {len(data_to_append)} rows to the 'Output' worksheet.")


Successfully appended 24 rows to the 'Output' worksheet.
